In [2]:
from IPython.core.display import display, HTML
display(HTML("<style>.jp-Cell { width: 140% !important; }</style>"))

/tmp/ipykernel_21467/3693427796.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from gensim.models import Word2Vec
from scipy.linalg import orthogonal_procrustes
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings('ignore')

# # =============================================================================
# # CONFIGURATION
# # =============================================================================

# Measure drift for target words
target_words = ['neural', 'memory', 'cell', 'agent','network','node']
target_words_guardian = ['cloud', 'stream', 'port']


# SEEDS        = [42, 123, 777]
# # SEEDS        = [777]
# N_ANCHORS    = 3000   # How many anchor words to use for alignment.
#                        # More = better alignment but slower.

# # File paths — adjust if your .bin files live elsewhere
# ARXIV_FILES = {
#     '1990-2011': {42: './CS_arXiv/u_model_1990_42.bin',  123: './CS_arXiv/u_model_1990_123.bin',  777: './CS_arXiv/u_model_1990_777.bin'},
#     '2012-2019': {42: './CS_arXiv/u_model_2012_42.bin',  123: './CS_arXiv/u_model_2012_123.bin',  777: './CS_arXiv/u_model_2012_777.bin'},
#     '2020-2023': {42: './CS_arXiv/u_model_2020_42.bin',  123: './CS_arXiv/u_model_2020_123.bin',  777: './CS_arXiv/u_model_2020_777.bin'},
#     # '2000-2009': {777: 'model_2000_777.bin'},
#     # '2012-2019': {777: 'model_2012_777.bin'},
#     # '2020-2025': {777: 'model_2020_777.bin'},

# }
# PUBMED_FILES = {
#     42:  './pubmed/model_pubmed_200_42.bin',
#     123: './pubmed/model_pubmed_200_123.bin',
#     777: './pubmed/model_pubmed_200_777.bin',
# }

# GUARDIAN_FILES = {
#     42:  './guardian/guardian_42.bin',
#     123: './guardian/guardian_123.bin',
#     777: './guardian/guardian_777.bin',
# }

# # NEWS_FILES = {
# #     42:  'news_corpus_description_42.bin',
# #     123: 'news_corpus_description_123.bin',
# #     777: 'news_corpus_description_777.bin',
# # }

# PERIOD_LABELS = list(ARXIV_FILES.keys())
# TRANSITIONS   = [
#     ('1990-2011', '2012-2019', '1990 → 2010s'),
#     ('2012-2019', '2020-2023', '2010s → 2020s'),
# ]

In [2]:
def get_anchors(model_a, model_b, exclude_words=None, min_count=500):
    """
    Compute the shared anchor word list between two models.
    Separated from alignment so the SAME anchor set can be reused
    across multiple alignment calls — critical for fair SNR comparison.
    """
    exclude = set(exclude_words or [])
    vocab_a  = set(model_a.wv.index_to_key)
    vocab_b  = set(model_b.wv.index_to_key)
    shared   = sorted(vocab_a & vocab_b - exclude)
    shared   = [w for w in shared
                if model_a.wv.get_vecattr(w, "count") > min_count
                and model_b.wv.get_vecattr(w, "count") > min_count]
    return shared


def fit_procrustes(model_a, model_b, anchors):
    """
    Fit rotation R using the given anchor list.
    Returns R, src_mean — the two things needed to transform
    any source vector into the target space.
    """
    A = np.array([model_a.wv[w] for w in anchors])
    B = np.array([model_b.wv[w] for w in anchors])

    # Normalise
    A /= np.linalg.norm(A, axis=1, keepdims=True)
    B /= np.linalg.norm(B, axis=1, keepdims=True)

    src_mean = A.mean(axis=0)
    A -= src_mean
    B -= B.mean(axis=0)

    R, _ = orthogonal_procrustes(A, B)

    cos_errors = [
        1 - np.dot((A[i] @ R), B[i]) /
        (np.linalg.norm(A[i] @ R) * np.linalg.norm(B[i]))
        for i in range(len(anchors))
    ]
    residual = np.mean(cos_errors)

    return R, src_mean, residual


def apply_procrustes(source_model, R, src_mean):
    """
    Apply a pre-fitted rotation to ALL vectors in source_model.
    Applies the same normalise → centre → rotate pipeline
    that was used during fitting.
    """
    all_words   = source_model.wv.index_to_key
    all_vectors = np.array([source_model.wv[w] for w in all_words])

    norms       = np.linalg.norm(all_vectors, axis=1, keepdims=True)
    norms       = np.where(norms == 0, 1, norms)
    all_vectors = all_vectors / norms

    all_vectors -= src_mean

    aligned = all_vectors @ R
    return aligned

In [3]:
# ── 1. Load models ─────────────────────────────────────────────────────────────
arxiv_2020_42 = Word2Vec.load("./CS_arXiv/u_model_2020_42.bin")
arxiv_2012_42 = Word2Vec.load("./CS_arXiv/u_model_2012_42.bin")
arxiv_1990_42 = Word2Vec.load("./CS_arXiv/u_model_1990_42.bin")
pubmed_42     = Word2Vec.load("./pubmed/model_pubmed_200_42.bin")
guardian_42   = Word2Vec.load("./guardian/guardian_42.bin")
print('loading models....')

# ── 1. Load models 123 ─────────────────────────────────────────────────────────────
arxiv_2020_123 = Word2Vec.load("./CS_arXiv/u_model_2020_123.bin")
arxiv_2012_123 = Word2Vec.load("./CS_arXiv/u_model_2012_123.bin")
arxiv_1990_123 = Word2Vec.load("./CS_arXiv/u_model_1990_123.bin")
pubmed_123     = Word2Vec.load("./pubmed/model_pubmed_200_123.bin")
guardian_123   = Word2Vec.load("./guardian/guardian_123.bin")
print('loading models....')

# ── 1. Load models 777 ─────────────────────────────────────────────────────────────
arxiv_2020_777 = Word2Vec.load("./CS_arXiv/u_model_2020_777.bin")
arxiv_2012_777 = Word2Vec.load("./CS_arXiv/u_model_2012_777.bin")
arxiv_1990_777 = Word2Vec.load("./CS_arXiv/u_model_1990_777.bin")
pubmed_777    = Word2Vec.load("./pubmed/model_pubmed_200_777.bin")
guardian_777 = Word2Vec.load("./guardian/guardian_777.bin")

print('finished loading models....')

loading models....
loading models....
finished loading models....


In [4]:
exclude = set(target_words + target_words_guardian)

def start_alignment(arxiv_42, arxiv_123, arxiv_777, target_model_42, target_model_123, target_model_777):

    cross_anchors = get_anchors(arxiv_42, target_model_42, exclude)
    
    noise_anchors = [w for w in cross_anchors
                          if w in arxiv_42.wv
                          and w in arxiv_123.wv
                          and arxiv_42.wv.get_vecattr(w, "count") > 500
                          and arxiv_123.wv.get_vecattr(w, "count") > 500]
    
    _, _, eps_42_123 = fit_procrustes(arxiv_42,  arxiv_123, noise_anchors)
    _, _, eps_42_777 = fit_procrustes(arxiv_42,  arxiv_777, noise_anchors)
    _, _, eps_123_777= fit_procrustes(arxiv_123, arxiv_777, noise_anchors)
    epsilon = np.mean([eps_42_123, eps_42_777, eps_123_777])
    
    R_42,  mean_42,  res_42  = fit_procrustes(arxiv_42,  target_model_42,  cross_anchors)
    R_123, mean_123, res_123 = fit_procrustes(arxiv_123, target_model_123, cross_anchors)
    R_777, mean_777, res_777 = fit_procrustes(arxiv_777, target_model_777, cross_anchors)
    
    cross_residual = np.mean([res_42, res_123, res_777])
    snr = cross_residual / epsilon
    
    print(f"Anchors used       : {len(cross_anchors)}")
    print(f"Noise floor ε      : {epsilon:.4f}")
    print(f"Cross-domain resid : {cross_residual:.4f}")
    print(f"SNR                : {snr:.2f}×")
    
    # ── Apply to get aligned vectors for drift scoring ─────────────────────────────
    aligned_model_42  = apply_procrustes(arxiv_42,  R_42,  mean_42)
    aligned_model_123 = apply_procrustes(arxiv_123, R_123, mean_123)
    aligned_model_777 = apply_procrustes(arxiv_777, R_777, mean_777)

    return aligned_model_42, aligned_model_123, aligned_model_777

In [5]:
aligned_pubmed_1990_42, aligned_pubmed_1990_123, aligned_pubmed_1990_777 = start_alignment(arxiv_1990_42, arxiv_1990_123, arxiv_1990_777, pubmed_42, pubmed_123, pubmed_777)
aligned_pubmed_2012_42, aligned_pubmed_2012_123, aligned_pubmed_2012_777 = start_alignment(arxiv_2012_42, arxiv_2012_123, arxiv_2012_777, pubmed_42, pubmed_123, pubmed_777)
aligned_pubmed_2020_42, aligned_pubmed_2020_123, aligned_pubmed_2020_777 = start_alignment(arxiv_2020_42, arxiv_2020_123, arxiv_2020_777, pubmed_42, pubmed_123, pubmed_777)

Anchors used       : 511
Noise floor ε      : 0.1143
Cross-domain resid : 0.3970
SNR                : 3.47×
Anchors used       : 1660
Noise floor ε      : 0.0987
Cross-domain resid : 0.5139
SNR                : 5.21×
Anchors used       : 1882
Noise floor ε      : 0.0836
Cross-domain resid : 0.5149
SNR                : 6.16×


In [6]:
aligned_guardian_1990_42, aligned_guardian_1990_123, aligned_guardian_1990_777 = start_alignment(arxiv_1990_42, arxiv_1990_123, arxiv_1990_777, guardian_42, guardian_123, guardian_777)
aligned_guardian_2012_42, aligned_guardian_2012_123, aligned_guardian_2012_777 =start_alignment(arxiv_2012_42, arxiv_2012_123, arxiv_2012_777, guardian_42, guardian_123, guardian_777)
aligned_guardian_2020_42, aligned_guardian_2020_123, aligned_guardian_2020_777 = start_alignment(arxiv_2020_42, arxiv_2020_123, arxiv_2020_777, guardian_42, guardian_123, guardian_777)

Anchors used       : 556
Noise floor ε      : 0.1165
Cross-domain resid : 0.4156
SNR                : 3.57×
Anchors used       : 2120
Noise floor ε      : 0.1021
Cross-domain resid : 0.5229
SNR                : 5.12×
Anchors used       : 2542
Noise floor ε      : 0.0878
Cross-domain resid : 0.5189
SNR                : 5.91×


In [7]:
def format_score(score):
    return f"{score:.4f}" if score else "N/A"
    
def cross_domain_distance(word, aligned_matrix, aligned_vocab, target_model):
    """Cosine distance between aligned source vector and target vector."""
    if word not in aligned_vocab or word not in target_model.wv:
        return None
    idx        = aligned_vocab.index(word)
    src_vec    = aligned_matrix[idx]
    tgt_vec    = target_model.wv[word]
    return cosine(src_vec, tgt_vec)

In [15]:
# Measure drift for target words
target_words = ['neural', 'memory', 'cell', 'agent','network','node']
target_words_guardian = ['cloud','stream' ,'port']

print("\n── Cross-domain drift (arXiv 1990 → PubMed) ──────────────")
print(f"{'Word':<15} {'seed':>10} {'→ 1990':>15} {'→ 2012':>15} {'→ 2020':>15}")
for w in target_words:
    print('--'*39)

    d_pm_1990_42 = cross_domain_distance(w, aligned_pubmed_1990_42, list(arxiv_1990_42.wv.index_to_key), pubmed_42)
    d_pm_1990_123 = cross_domain_distance(w, aligned_pubmed_1990_123, list(arxiv_1990_123.wv.index_to_key), pubmed_123)
    d_pm_1990_777 = cross_domain_distance(w, aligned_pubmed_1990_777, list(arxiv_1990_777.wv.index_to_key), pubmed_777)

    d_pm_2012_42 = cross_domain_distance(wAstrid S , aligned_pubmed_2012_42, list(arxiv_2012_42.wv.index_to_key), pubmed_42)
    d_pm_2012_123 = cross_domain_distance(w, aligned_pubmed_2012_123, list(arxiv_2012_123.wv.index_to_key), pubmed_123)
    d_pm_2012_777 = cross_domain_distance(w, aligned_pubmed_2012_777, list(arxiv_2012_777.wv.index_to_key), pubmed_777)

    d_pm_2020_42 = cross_domain_distance(w, aligned_pubmed_2020_42, list(arxiv_2020_42.wv.index_to_key), pubmed_42)
    d_pm_2020_123 = cross_domain_distance(w, aligned_pubmed_2020_123, list(arxiv_2020_123.wv.index_to_key), pubmed_123)
    d_pm_2020_777 = cross_domain_distance(w, aligned_pubmed_2020_777, list(arxiv_2020_777.wv.index_to_key), pubmed_777)

    print(f"{w:<15} {42:>9} {format_score(d_pm_1990_42):>16} {format_score(d_pm_2012_42):>15} {format_score(d_pm_2020_42):>15}")
    print(f"{w:<15} {123:>10} {format_score(d_pm_1990_123):>15} {format_score(d_pm_2012_123):>15} {format_score(d_pm_2020_123):>15}")
    print(f"{w:<15} {777:>10} {format_score(d_pm_1990_777):>15} {format_score(d_pm_2012_777):>15} {format_score(d_pm_2020_777):>15}")
    

print("\n── Cross-domain drift (arXiv 1990 → Guardian) ──────────────")
print(f"{'Word':<15} {'seed':>10} {'→ 1990':>15} {'→ 2012':>15} {'→ 2020':>15}")
for w in target_words_guardian:
    print('--'*39)

    d_gd_1990_42 = cross_domain_distance(w, aligned_guardian_1990_42, list(arxiv_1990_42.wv.index_to_key), guardian_42)
    d_gd_1990_123 = cross_domain_distance(w, aligned_guardian_1990_123, list(arxiv_1990_123.wv.index_to_key), guardian_123)
    d_gd_1990_777 = cross_domain_distance(w, aligned_guardian_1990_777, list(arxiv_1990_777.wv.index_to_key), guardian_777)

    d_gd_2012_42 = cross_domain_distance(w, aligned_guardian_2012_42, list(arxiv_2012_42.wv.index_to_key), guardian_42)
    d_gd_2012_123 = cross_domain_distance(w, aligned_guardian_2012_123, list(arxiv_2012_123.wv.index_to_key), guardian_123)
    d_gd_2012_777 = cross_domain_distance(w, aligned_guardian_2012_777, list(arxiv_2012_777.wv.index_to_key), guardian_777)

    d_gd_2020_42 = cross_domain_distance(w, aligned_guardian_2020_42, list(arxiv_2020_42.wv.index_to_key), guardian_42)
    d_gd_2020_123 = cross_domain_distance(w, aligned_guardian_2020_123, list(arxiv_2020_123.wv.index_to_key), guardian_123)
    d_gd_2020_777 = cross_domain_distance(w, aligned_guardian_2020_777, list(arxiv_2020_777.wv.index_to_key), guardian_777)

    print(f"{w:<15} {42:>9} {format_score(d_gd_1990_42):>16} {format_score(d_gd_2012_42):>15} {format_score(d_gd_2020_42):>15}")
    print(f"{w:<15} {123:>10} {format_score(d_gd_1990_123):>15} {format_score(d_gd_2012_123):>15} {format_score(d_gd_2020_123):>15}")
    print(f"{w:<15} {777:>10} {format_score(d_gd_1990_777):>15} {format_score(d_gd_2012_777):>15} {format_score(d_gd_2020_777):>15}")




── Cross-domain drift (arXiv 1990 → PubMed) ──────────────
Word                  seed          → 1990          → 2012          → 2020
------------------------------------------------------------------------------
neural                 42           1.0334          0.8359          0.9475
neural                 123          0.9523          0.8543          0.8085
neural                 777          0.9856          0.8508          0.8349
------------------------------------------------------------------------------
memory                 42           0.8887          0.9280          0.8798
memory                 123          1.0136          0.9790          0.9207
memory                 777          0.9733          0.9265          0.9114
------------------------------------------------------------------------------
cell                   42           0.8575          0.7331          0.6354
cell                   123          0.8654          0.7751          0.6777
cell                   777  

In [13]:
import numpy as np

def drift_across_seeds(word, aligned_matrices, vocab, target_model):
    scores = []
    for mat in aligned_matrices:
        d = cross_domain_distance(word, mat, vocab, target_model)
        if d is not None:
            scores.append(d)
    if len(scores) == 0:
        return None, None
    return np.mean(scores), np.std(scores)
    
aligned = [
    aligned_pubmed_1990_42,
    aligned_pubmed_1990_123,
    aligned_pubmed_1990_777
]

# aligned = [
#     aligned_pubmed_2012_42,
#     aligned_pubmed_2012_123,
#     aligned_pubmed_2012_777
# ]

# aligned = [
#     aligned_pubmed_2020_42,
#     aligned_pubmed_2020_123,
#     aligned_pubmed_2020_777
# ]


vocab = list(arxiv_2012_42.wv.index_to_key)
vocab = list(arxiv_1990_42.wv.index_to_key)
# vocab = list(arxiv_2020_42.wv.index_to_key)

for w in target_words:
    mean_drift, std_drift = drift_across_seeds(
        w, aligned, vocab, pubmed_42
    )
    print(f"{w}: mean={mean_drift:.4f}, std={std_drift:.4f}")

neural: mean=0.9965, std=0.0382
memory: mean=0.9802, std=0.0648
cell: mean=0.9445, std=0.0679
agent: mean=0.9667, std=0.0921
network: mean=0.9361, std=0.0255
node: mean=0.9002, std=0.2001
port: mean=0.9813, std=0.0530


In [58]:
import random

def sample_baseline_words(vocab, target_words, n=100):
    candidates = list(set(vocab) - set(target_words))
    return random.sample(candidates, n)
baseline_words = sample_baseline_words(vocab, target_words, n=100)

target_drifts = []

for w in target_words:
    mean_drift, _ = drift_across_seeds(
        w, aligned, vocab, pubmed_42
    )
    if mean_drift is not None:
        target_drifts.append(mean_drift)

baseline_drifts = []

for w in baseline_words:
    mean_drift, _ = drift_across_seeds(
        w, aligned, vocab, pubmed_42
    )
    if mean_drift is not None:
        baseline_drifts.append(mean_drift)

In [59]:
from scipy.stats import mannwhitneyu

stat, p_value = mannwhitneyu(
    target_drifts,
    baseline_drifts,
    alternative='greater'   # key: testing if target > baseline
)

print(f"U-statistic: {stat}")
print(f"p-value: {p_value}")

U-statistic: 225.0
p-value: 0.7151644476216984


In [60]:
print("Target mean drift:", np.mean(target_drifts))
print("Baseline mean drift:", np.mean(baseline_drifts))

Target mean drift: 0.95794827
Baseline mean drift: 0.9647262
